# 2. Ay: CNN vs Transformer kiyaslamasi (5-fold cross validation)

**Plan (2. ay):** U-Net, U-Net++, LinkNet (CNN) ve ViT, Swin Transformer modellerinin veri setine uyarlanmasi;
ham ve optimum odakli goruntulerle egitim; Dice / Focal / BCE loss karsilastirmasi.

**Once:** `Runtime > Change runtime type > T4 GPU`.
Sonuclar her fold bittiginde Drive'a yazilir. Oturum koparsa hucreleri bastan calistirin; bitmis fold'lar atlanir, kaldigi yerden devam eder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1) Ayarlar

In [ ]:
CODE_DIR = '/content/drive/MyDrive/unet_segmentation'     # benchmark.py burada olmali
SEG      = '/content/drive/MyDrive/SEG'
MASK_DIR = f'{SEG}/maske'
DATA = {                                                    # veri gruplari
    'ham':     f'{SEG}/train',
    'net':     f'{SEG}/test_net',
    'bulanik': f'{SEG}/test_bulanık',
}
OUT = f'{CODE_DIR}/kiyaslama'                               # results.csv, summary.csv, loglar

import os
for p in [CODE_DIR, MASK_DIR, *DATA.values()]:
    print('OK ' if os.path.isdir(p) else 'YOK', p)
print('OK ' if os.path.exists(f'{CODE_DIR}/benchmark.py') else 'YOK', 'benchmark.py')

## 2) Kurulum

In [ ]:
%cd {CODE_DIR}
!pip install -q segmentation-models-pytorch timm nibabel
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3) Veri gruplarini kontrol et
Hangi klasorun "optimum odakli" oldugunu dogrulamak icin: ayni goruntunun gruplar arasi farki ve keskinligi
(Laplacian varyansi; yuksek = daha keskin). Ayrica tum gruplarda ortak olan goruntuler bulunur;
adil kiyas icin tum deneyler AYNI goruntuleri ve AYNI fold bolunmesini kullanir.

In [ ]:
import sys, numpy as np, cv2
sys.path.insert(0, CODE_DIR)
from benchmark import list_files, load_rgb

EXTS = ('.png', '.nii', '.nii.gz')
files = {k: list_files(v, EXTS) for k, v in DATA.items()}
masks = list_files(MASK_DIR, ('.nii', '.nii.gz'))
common = set(masks).intersection(*[set(f) for f in files.values()])
IDS = ','.join(sorted(common, key=int))
print({k: len(v) for k, v in files.items()}, '| maske:', len(masks), '| ortak:', len(common))

def sharp(img):
    return cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()

for n in sorted(common, key=int)[:3]:
    imgs = {k: load_rgb(files[k][n]) for k in DATA}
    line = f'goruntu {n}: ' + ' | '.join(f'{k} keskinlik={sharp(v):.0f}' for k, v in imgs.items())
    line += ' | fark(ham-net)=%.2f fark(ham-bulanik)=%.2f' % (
        np.abs(imgs['ham'].astype(int) - imgs['net']).mean(), np.abs(imgs['ham'].astype(int) - imgs['bulanik']).mean())
    print(line)

## 4) Deney 1: Ham goruntuler, tum modeller
6 model x 5 fold. Her fold'da 72 goruntuyle egitim, 18 goruntuyle test.
Ilk calistirmada veri Drive'dan okunup cache'lenir (birkac dakika).

In [ ]:
COMMON = f'--ids {IDS} --masks "{MASK_DIR}" --out "{OUT}" --epochs 30 --folds 5'
!python benchmark.py --data-name ham --images "{DATA['ham']}" {COMMON} --loss bce_dice     --models unet_basic unet unetpp linknet vit swin

## 5) Deney 2: Optimum odakli (net) goruntuler, ayni modeller

In [ ]:
!python benchmark.py --data-name net --images "{DATA['net']}" {COMMON} --loss bce_dice     --models unet_basic unet unetpp linknet vit swin

## 6) Deney 3: Loss fonksiyonu karsilastirmasi
Bir CNN (U-Net) ve bir Transformer (Swin) uzerinde; `bce_dice` sonuclari Deney 1'den gelir.

In [ ]:
for loss in ['bce', 'dice', 'focal', 'focal_dice']:
    !python benchmark.py --data-name ham --images "{DATA['ham']}" {COMMON} --loss {loss} --models unet swin

## 7) (Opsiyonel) Bulanik goruntuler

In [ ]:
!python benchmark.py --data-name bulanik --images "{DATA['bulanik']}" {COMMON} --loss bce_dice     --models unet linknet vit swin

## 8) Sonuclar

In [ ]:
!python benchmark.py --summary --out "{OUT}"

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f'{OUT}/results.csv')

# Model x veri grubu (bce_dice), 5 fold ortalamasi +- std
d = df[df.loss == 'bce_dice'].groupby(['model', 'data']).dice.agg(['mean', 'std']).unstack('data')
order = [m for m in ['unet_basic', 'unet', 'unetpp', 'linknet', 'vit', 'swin'] if m in d.index]
d = d.loc[order]
ax = d['mean'].plot.bar(yerr=d['std'], capsize=3, figsize=(10, 4), rot=0)
ax.set_ylabel('Dice (5-fold ort.)'); ax.set_ylim(d['mean'].min().min() - 0.05, 1.0)
ax.set_title('Model ve veri grubuna gore Dice'); plt.tight_layout()
plt.savefig(f'{OUT}/dice_model_veri.png', dpi=150); plt.show()

# Loss karsilastirmasi (ham)
l = df[df.data == 'ham'].groupby(['loss', 'model']).dice.agg(['mean', 'std']).unstack('model')
if len(l) > 1:
    ax = l['mean'].plot.bar(yerr=l['std'], capsize=3, figsize=(8, 4), rot=0)
    ax.set_ylabel('Dice (5-fold ort.)'); ax.set_ylim(l['mean'].min().min() - 0.05, 1.0)
    ax.set_title('Loss fonksiyonu karsilastirmasi (ham veri)'); plt.tight_layout()
    plt.savefig(f'{OUT}/dice_loss.png', dpi=150); plt.show()